# Neuron Atlas Interface in Google Colab

Suche, Druckansicht, API und Geometrie-Werkzeuge für Goodfires
[Qwen3.6 Neuron Atlas](https://www.goodfire.com/research/public_neuron_atlas) – ohne lokale Installation.
Die Zellen der Reihe nach ausführen (▶ oder Strg+Enter).

> **Inoffiziell.** Die Daten gehören Goodfire AI. Ihre Nutzungsbedingungen erlauben persönliche,
> nicht-kommerzielle Nutzung; automatisiertes Kopieren und Weiterverbreiten nur mit Zustimmung.
> Das Werkzeug lädt deshalb nur, was du öffnest oder ausdrücklich herunterlädst – gedrosselt
> (4 Anfragen/s) und jede Datei nur einmal. Mehr im
> [README](https://github.com/Erikiss/Interface-Neuron-Atlas-by-GoodFire-AI/blob/claude/neuronatlas-interface-api-h1uw92/README.md#rechtliches-und-rücksicht).

## 1. Installieren

Etwa eine Minute. Meldet pip, dass Pakete von Colab ersetzt wurden, und verlangt Colab einen
Neustart der Sitzung, diesen bestätigen und mit Schritt 2 weitermachen.

In [ ]:
%pip install -q "neuron-atlas-interface[all] @ git+https://github.com/Erikiss/Interface-Neuron-Atlas-by-GoodFire-AI.git@claude/neuronatlas-interface-api-h1uw92"

## 2. (optional) Den Spiegel in Google Drive aufbewahren

Eine Colab-Sitzung ist vorübergehend: Was heruntergeladen wurde, ist beim nächsten Mal weg.
Mit `USE_DRIVE = True` wird ein früher gesicherter Spiegel aus Drive zurückgeholt; am Ende
(Schritt 8) wird er wieder gesichert – als **eine** Archivdatei, denn tausende kleine Dateien und
die SQLite-Datenbanken direkt auf Drive zu betreiben ist langsam und unzuverlässig.

In [ ]:
USE_DRIVE = False
DRIVE_DIR = "/content/drive/MyDrive/neuron-atlas"

if USE_DRIVE:
    import glob
    from google.colab import drive
    from neuron_atlas import notebook

    drive.mount("/content/drive")
    for archive in sorted(glob.glob(f"{DRIVE_DIR}/neuron-atlas-*.tar")):
        print(archive, "->", notebook.restore(archive), "Dateien zurückgeholt")

## 3. Die Oberfläche starten

Startet Weboberfläche, REST-API (`/docs`) und MCP-Endpunkt im Hintergrund dieses Notebooks und
zeigt einen Link, der die Oberfläche in einem neuen Tab öffnet (nur für dich sichtbar). Beim ersten
Start werden sieben Index-Dateien geladen (zusammen etwa 3 MB).

In [ ]:
from neuron_atlas import notebook

server = notebook.start()
atlas = server.atlas          # derselbe Atlas für die Python-Zellen unten
atlas.info()["atlas"]

## 4. Aus Python suchen

Dieselbe Syntax wie in der Suchzeile: Wörter, `"Phrasen"`, `-ausschließen`, `präfix*`, `OR` / `NOT`
(in Großbuchstaben), Filter wie `layer:24..36 cat:mono,fn pole:neg sim:>0.8`.

In [ ]:
import pandas as pd

res = atlas.search({"q": "braille OR tactile", "limit": 20})
print(f"{res.total} Treffer")
pd.DataFrame([{"Pol": h.pid, "Kategorie": h.category_words, "Beschreibung": h.description} for h in res.hits])

In [ ]:
# Ein Pol als Dokument (dieselbe Ansicht wie die Druckseite); fehlende Dateien werden einmal geholt
notebook.show_pole("L30_E47_N449__pos", include=("modes", "examples", "overview"), max_examples=1)

In [ ]:
# Ähnliche Bedeutungen - zu einem Text oder zu einem Pol
sim = atlas.similar(text="random hex hashes and base64", top=10)
pd.DataFrame([{"Pol": i.pid, "Modus": i.mode_index, "Ähnlichkeit": round(i.score, 3), "Text": i.label or i.description} for i in sim.items])

## 5. Eine semantische Region herunterladen

Erst ansehen, was geladen würde, dann laden. Danach sind auch die langen Texte der Region
durchsuchbar (Erklärungen, Subcircuits, Reports) und die Geometrie-Werkzeuge haben mehr Daten.

In [ ]:
from neuron_atlas.models import MirrorRequest, Selection

request = MirrorRequest(selection=Selection(similar_to_text="braille and dot patterns", top=10))
atlas.plan_mirror(request).estimate()

In [ ]:
plan, report = atlas.mirror(request)    # gedrosselt, jederzeit abbrechbar (■), setzt beim nächsten Mal fort
report.as_dict()

## 6. Geometrie: Karten und Tabellen

Die Karte ordnet Modi nach Bedeutung an (UMAP); Farbe = Cluster. In
[docs/geometry.md](https://github.com/Erikiss/Interface-Neuron-Atlas-by-GoodFire-AI/blob/claude/neuronatlas-interface-api-h1uw92/docs/geometry.md)
steht, welche Strukturen in den Daten stecken und welche nicht.

In [ ]:
from neuron_atlas.geometry.maps import semantic_map
from neuron_atlas.models import SearchQuery

pids = [h.pid for h in atlas.iter_search(SearchQuery(q="hash OR base64 OR hex"))]
data = semantic_map(atlas, pids, unit="mode", method="umap", clusters=8)
print(len(data["points"]), "Punkte;", "; ".join(data["notes"]))
notebook.show_map(data, title="Hashes, Base64, Hex")

In [ ]:
# Alles Gespiegelte als Tabellen (eine Zeile pro Feuerung, Modus×Modus-Ähnlichkeiten, Circuit-Kanten, …)
from neuron_atlas.geometry.tables import write_tables

written = write_tables(atlas, "/content/tables", fmt="parquet")
print(written)
pd.read_parquet("/content/tables/modes.parquet").head()

## 7. Kommandozeile im Notebook

Alle Befehle des README gehen auch hier, mit `!` davor.

In [ ]:
!neuron-atlas search "chess opening" -n 5

## 8. (optional) Den Spiegel in Drive sichern

Eine Archivdatei pro Atlas-Version; beim nächsten Mal holt Schritt 2 sie zurück.

In [ ]:
if USE_DRIVE:
    print(notebook.backup(DRIVE_DIR))

## Chatbots

Der MCP-Endpunkt läuft mit (`/mcp`), ist aber nur über Colabs angemeldeten Proxy erreichbar – also
für dich im Browser, nicht für einen Chatbot im Netz. Für Claude Desktop oder Claude Code das Paket
lokal installieren und `neuron-atlas mcp` eintragen (siehe README, „Chatbots anbinden“).